# **Análisis Avanzado de Terremotos en Chile (2000-2024) - Fase 4**

**Nombres**: Ademir Ortiz, Bruno Sepúlveda Manrique, Lucas Bravo  
**Grupo**: 9  
**Curso**: Programación para la Ciencia de Datos  
**Profesor**: Omar Salinas Sanchez

---

## **Introducción**

Este notebook implementa **análisis exploratorio avanzado, visualizaciones geoespaciales y modelos predictivos** sobre los datos sísmicos de Chile (2000-2024) procesados mediante la arquitectura POO de Fase 3.

**Fases del Análisis**:
1. Carga del pipeline POO de Fase 3 para obtener datos limpios y normalizados
2. Análisis exploratorio univariado (distribuciones, outliers, estadísticas)
3. Análisis bivariado (correlaciones, regresiones, ANOVA)
4. Series temporales (tendencias, cambios estructurales, clustering)
5. Análisis geoespacial (mapas interactivos, densidades, regiones)
6. Perfiles de riesgo sísmico por región
7. Modelos predictivos de magnitud (ML)
8. Dashboard de KPIs y métricas clave

# **PARTE 1: IMPORTACIÓN DE LIBRERÍAS AVANZADAS**

## **1.1 Librerías Científicas y de Análisis**

In [ ]:
# ============================================================================
# IMPORTACIÓN DE LIBRERÍAS AVANZADAS
# ============================================================================

# Análisis de datos y manipulación
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.spatial.distance import cdist
from scipy import signal

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import HeatMap, MarkerCluster

# Geoespacial
from geopy.distance import geodesic

# Tiempo
import time
from datetime import datetime

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('✓ Todas las librerías cargadas correctamente')

# **PARTE 2: CARGA DEL PIPELINE POO DE FASE 3**

## **2.1 Definición de Clases Transformadoras (desde Fase 3)**

In [ ]:
# ============================================================================
# DEFINICIÓN DE CLASES TRANSFORMADORAS (Copiadas de Fase 3)
# ============================================================================

import logging
from abc import ABC, abstractmethod
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
import re
from datetime import datetime

# Configurar logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# [Las clases Transformer, CargadorDatos, RenombradoVariables, etc. 
#  se incluirían aquí si se necesitara redefinir, pero se asume que están
#  disponibles de Fase 3 o se cargan desde archivo externo]

# Para este notebook, importaremos el DataFrame procesado directamente
RUTA_DATASET = r'../../data/raw/EarthquakesChile_2000-2024.csv'
print(f'✓ Ruta del dataset configurada: {RUTA_DATASET}')

## **2.2 Ejecución del Pipeline POO y Carga de Datos Procesados**

In [ ]:
# ============================================================================
# CARGA Y PROCESAMIENTO DEL DATASET
# ============================================================================

# Para este notebook de Fase 4, cargamos directamente el CSV
# (En un proyecto completo, se importaría el pipeline de Fase 3)

print('Cargando dataset de terremotos Chile 2000-2024...')
df_raw = pd.read_csv(RUTA_DATASET, encoding='utf-8')

print(f'\n✓ Dataset cargado exitosamente')
print(f'  - Dimensiones: {df_raw.shape[0]:,} registros × {df_raw.shape[1]} columnas')
print(f'  - Memoria: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'\nPrimeras 5 filas:')
display(df_raw.head())
print(f'\nInformación del dataset:')
df_raw.info()

## **2.3 Procesamiento Inicial (Emulando Fase 3)**

In [ ]:
# ============================================================================
# PROCESAMIENTO INICIAL DEL DATASET
# ============================================================================

# Crear DataFrame procesado aplicando transformaciones básicas
df = df_raw.copy()

# 1. Renombramiento de columnas
mapeo_columnas = {
    'UTC Date': 'FECHA_UTC',
    'Profundidad': 'PROFUNDIDAD_TEXTO',
    'Magnitud': 'MAGNITUD_TEXTO',
    'Date': 'FECHA_LOCAL',
    'Hour': 'HORA_LOCAL',
    'Location': 'UBICACION',
    'Latitude': 'LATITUD',
    'Longitude': 'LONGITUD'
}
df = df.rename(columns=mapeo_columnas)

# 2. Procesamiento de fechas
df['FECHA_UTC'] = pd.to_datetime(df['FECHA_UTC'], errors='coerce')
df['ANIO'] = df['FECHA_UTC'].dt.year
df['MES'] = df['FECHA_UTC'].dt.month
df['DIA'] = df['FECHA_UTC'].dt.day

# 3. Extracción de profundidad
df['PROFUNDIDAD_KM'] = df['PROFUNDIDAD_TEXTO'].str.extract(r'([0-9.]+)').astype(float)

# 4. Extracción de magnitud
df[['MAGNITUD_VALOR', 'MAGNITUD_ESCALA']] = df['MAGNITUD_TEXTO'].str.extract(r'([0-9.]+)\s*([A-Za-z]+)')
df['MAGNITUD_VALOR'] = df['MAGNITUD_VALOR'].astype(float)

# 5. Estandarización a Mw (Scordilis 2006)
def convertir_a_mw(row):
    valor = row['MAGNITUD_VALOR']
    escala = row['MAGNITUD_ESCALA']
    
    if pd.isna(valor):
        return np.nan
    
    conversiones = {
        'Ml': 0.85 * valor + 1.03,
        'Mlv': 0.85 * valor + 1.03,
        'Mb': 0.85 * valor + 1.03,
        'Mc': 0.85 * valor + 1.03,
        'Ms': 0.67 * valor + 2.07,
        'Mw': valor,
        'Mww': valor,
        'M': valor
    }
    return conversiones.get(escala, valor)

df['MAGNITUD_MW'] = df.apply(convertir_a_mw, axis=1)

# 6. Filtro geográfico (solo Chile)
chile_lat_min, chile_lat_max = -56.0, -17.0
chile_lon_min, chile_lon_max = -90.0, -66.0
df = df[(df['LATITUD'] >= chile_lat_min) & (df['LATITUD'] <= chile_lat_max) &
        (df['LONGITUD'] >= chile_lon_min) & (df['LONGITUD'] <= chile_lon_max)].copy()

# 7. Eliminación de nulos en variables críticas
df = df.dropna(subset=['LATITUD', 'LONGITUD', 'MAGNITUD_MW', 'PROFUNDIDAD_KM', 'FECHA_UTC'])

# 8. Categorización
df['CATEGORIA_PROFUNDIDAD'] = pd.cut(df['PROFUNDIDAD_KM'],
                                      bins=[0, 70, 300, float('inf')],
                                      labels=['Superficial', 'Intermedio', 'Profundo'])

df['CATEGORIA_MAGNITUD'] = pd.cut(df['MAGNITUD_MW'],
                                   bins=[0, 2, 4, 5, 6, 7, 8, float('inf')],
                                   labels=['Micro', 'Menor', 'Ligero', 'Moderado', 'Fuerte', 'Mayor', 'Grande'])

print(f'✓ Dataset procesado exitosamente')
print(f'  - Registros: {len(df):,}')
print(f'  - Período: {df["ANIO"].min():.0f} - {df["ANIO"].max():.0f}')
print(f'  - Magnitud Mw: [{df["MAGNITUD_MW"].min():.2f}, {df["MAGNITUD_MW"].max():.2f}]')
print(f'  - Profundidad: [{df["PROFUNDIDAD_KM"].min():.1f}, {df["PROFUNDIDAD_KM"].max():.1f}] km')
print(f'\nMuestras de datos procesados:')
display(df.head(10))

# **PARTE 3: EXPLORACIÓN UNIVARIADA DE VARIABLES SÍSMICAS**

## **3.1 Análisis de Magnitud MW**

In [ ]:
# ============================================================================
# ANÁLISIS UNIVARIADO: MAGNITUD
# ============================================================================

print('ESTADÍSTICAS DESCRIPTIVAS - MAGNITUD MW')
print('=' * 80)
print(f'\nConteo: {df["MAGNITUD_MW"].count():,}')
print(f'Media: {df["MAGNITUD_MW"].mean():.3f}')
print(f'Mediana: {df["MAGNITUD_MW"].median():.3f}')
print(f'Desv. Est.: {df["MAGNITUD_MW"].std():.3f}')
print(f'Mín: {df["MAGNITUD_MW"].min():.3f}')
print(f'P25: {df["MAGNITUD_MW"].quantile(0.25):.3f}')
print(f'P75: {df["MAGNITUD_MW"].quantile(0.75):.3f}')
print(f'Máx: {df["MAGNITUD_MW"].max():.3f}')
print(f'Asimetría: {stats.skew(df["MAGNITUD_MW"].dropna()):.3f}')
print(f'Curtosis: {stats.kurtosis(df["MAGNITUD_MW"].dropna()):.3f}')

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histograma
axes[0, 0].hist(df['MAGNITUD_MW'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].set_title('Distribución de Magnitud MW', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Magnitud MW')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].grid(True, alpha=0.3)

# Boxplot
axes[0, 1].boxplot(df['MAGNITUD_MW'].dropna(), vert=True)
axes[0, 1].set_title('Boxplot de Magnitud MW', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Magnitud MW')
axes[0, 1].grid(True, alpha=0.3)

# Q-Q plot
stats.probplot(df['MAGNITUD_MW'].dropna(), dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normalidad)', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Violin plot por categoría
df_cat = df[df['CATEGORIA_MAGNITUD'].notna()]
sns.violinplot(data=df_cat, x='CATEGORIA_MAGNITUD', y='MAGNITUD_MW', ax=axes[1, 1], palette='husl')
axes[1, 1].set_title('Distribución por Categoría de Magnitud', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Categoría')
axes[1, 1].set_ylabel('Magnitud MW')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('analisis_magnitud.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Gráficos de magnitud generados')

## **3.2 Análisis de Profundidad**

In [ ]:
# ============================================================================
# ANÁLISIS UNIVARIADO: PROFUNDIDAD
# ============================================================================

print('ESTADÍSTICAS DESCRIPTIVAS - PROFUNDIDAD')
print('=' * 80)
print(f'\nConteo: {df["PROFUNDIDAD_KM"].count():,}')
print(f'Media: {df["PROFUNDIDAD_KM"].mean():.2f} km')
print(f'Mediana: {df["PROFUNDIDAD_KM"].median():.2f} km')
print(f'Desv. Est.: {df["PROFUNDIDAD_KM"].std():.2f} km')
print(f'Mín: {df["PROFUNDIDAD_KM"].min():.2f} km')
print(f'P25: {df["PROFUNDIDAD_KM"].quantile(0.25):.2f} km')
print(f'P75: {df["PROFUNDIDAD_KM"].quantile(0.75):.2f} km')
print(f'Máx: {df["PROFUNDIDAD_KM"].max():.2f} km')

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma con KDE
axes[0].hist(df['PROFUNDIDAD_KM'].dropna(), bins=50, edgecolor='black', alpha=0.6, color='coral', density=True, label='Histograma')
from scipy.stats import gaussian_kde
kde = gaussian_kde(df['PROFUNDIDAD_KM'].dropna())
x_range = np.linspace(df['PROFUNDIDAD_KM'].min(), df['PROFUNDIDAD_KM'].max(), 200)
axes[0].plot(x_range, kde(x_range), 'b-', linewidth=2, label='KDE')
axes[0].set_title('Distribución de Profundidad', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Profundidad (km)')
axes[0].set_ylabel('Densidad')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Violin plot por categoría
df_cat = df[df['CATEGORIA_PROFUNDIDAD'].notna()]
sns.violinplot(data=df_cat, x='CATEGORIA_PROFUNDIDAD', y='PROFUNDIDAD_KM', ax=axes[1], palette='Set2')
axes[1].set_title('Distribución por Categoría de Profundidad', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Categoría de Profundidad')
axes[1].set_ylabel('Profundidad (km)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('analisis_profundidad.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Gráficos de profundidad generados')

# **PARTE 4: ANÁLISIS BIVARIADO - RELACIONES ENTRE VARIABLES**

## **4.1 Matriz de Correlaciones**

In [ ]:
# ============================================================================
# ANÁLISIS BIVARIADO: CORRELACIONES
# ============================================================================

# Variables numéricas para correlación
variables_numericas = ['MAGNITUD_MW', 'PROFUNDIDAD_KM', 'LATITUD', 'LONGITUD', 'ANIO']
df_corr = df[variables_numericas].dropna()

# Matriz de correlación
corr_matrix = df_corr.corr(method='pearson')

print('MATRIZ DE CORRELACIONES - MÉTODO PEARSON')
print('=' * 80)
print(corr_matrix.round(3))

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Matriz de Correlaciones de Variables Sísmicas', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('matriz_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Matriz de correlaciones generada')

## **4.2 Scatter Plots de Relaciones Clave**

In [ ]:
# ============================================================================
# SCATTER PLOTS: MAGNITUD vs PROFUNDIDAD
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot 1: Magnitud vs Profundidad
scatter1 = axes[0].scatter(df['PROFUNDIDAD_KM'], df['MAGNITUD_MW'], 
                           c=df['ANIO'], cmap='viridis', s=30, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[0].set_title('Magnitud MW vs Profundidad (coloreado por año)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Profundidad (km)')
axes[0].set_ylabel('Magnitud MW')
axes[0].grid(True, alpha=0.3)
cbar1 = plt.colorbar(scatter1, ax=axes[0])
cbar1.set_label('Año')

# Regresión LOWESS
from scipy.signal import savgol_filter
sorted_idx = np.argsort(df['PROFUNDIDAD_KM'])
prof_sorted = df['PROFUNDIDAD_KM'].iloc[sorted_idx].values
mag_sorted = df['MAGNITUD_MW'].iloc[sorted_idx].values
if len(prof_sorted) > 10:
    mag_smooth = savgol_filter(mag_sorted, window_length=min(51, len(mag_sorted) - 1 if len(mag_sorted) % 2 == 0 else len(mag_sorted)), polyorder=3)
    axes[0].plot(prof_sorted, mag_smooth, 'r-', linewidth=2, label='Tendencia LOWESS')
    axes[0].legend()

# Scatter plot 2: Magnitud vs Latitud (geográfico)
scatter2 = axes[1].scatter(df['LATITUD'], df['MAGNITUD_MW'],
                           c=df['PROFUNDIDAD_KM'], cmap='plasma', s=30, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[1].set_title('Magnitud MW vs Latitud (coloreado por profundidad)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Latitud (°S)')
axes[1].set_ylabel('Magnitud MW')
axes[1].grid(True, alpha=0.3)
cbar2 = plt.colorbar(scatter2, ax=axes[1])
cbar2.set_label('Profundidad (km)')

plt.tight_layout()
plt.savefig('scatter_magnitud_profundidad.png', dpi=150, bbox_inches='tight')
plt.show()

print('✓ Scatter plots generados')

# **PARTE 5: SERIES TEMPORALES DE ACTIVIDAD SÍSMICA**

## **5.1 Tendencias Anuales de Sismicidad**

In [ ]:
# ============================================================================
# SERIES TEMPORALES: TENDENCIA ANUAL
# ============================================================================

# Agrupar por año
tendencia_anual = df.groupby('ANIO').agg({
        'MAGNITUD_MW': ['count', 'mean', 'max', 'std'],
        'PROFUNDIDAD_KM': 'mean'
    }).reset_index()

tendencia_anual.columns = ['Año', 'Eventos', 'Magnitud_Media', 'Magnitud_Max', 'Magnitud_Std', 'Profundidad_Media']

print('TENDENCIAS ANUALES DE SISMICIDAD')
print('=' * 80)
print(tendencia_anual.to_string(index=False))

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Gráfico 1: Número de eventos por año
axes[0, 0].bar(tendencia_anual['Año'], tendencia_anual['Eventos'], color='steelblue', edgecolor='black', alpha=0.7)
z = np.polyfit(tendencia_anual['Año'], tendencia_anual['Eventos'], 1)
p = np.poly1d(z)
axes[0, 0].plot(tendencia_anual['Año'], p(tendencia_anual['Año']), "r--", linewidth=2, label=f'Tendencia: y={z[0]:.2f}x+{z[1]:.0f}')
axes[0, 0].set_title('Número de Eventos Sísmicos Anuales', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Año')
axes[0, 0].set_ylabel('Cantidad de Eventos')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Gráfico 2: Magnitud máxima anual
axes[0, 1].plot(tendencia_anual['Año'], tendencia_anual['Magnitud_Max'], marker='o', linewidth=2, markersize=8, color='coral')
axes[0, 1].axhline(y=6.0, color='red', linestyle='--', linewidth=2, label='Umbral Mw 6.0')
axes[0, 1].set_title('Magnitud Máxima Anual', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Año')
axes[0, 1].set_ylabel('Magnitud MW')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Gráfico 3: Magnitud media anual
axes[1, 0].plot(tendencia_anual['Año'], tendencia_anual['Magnitud_Media'], marker='s', linewidth=2, markersize=6, color='mediumseagreen')
axes[1, 0].fill_between(tendencia_anual['Año'], 
                         tendencia_anual['Magnitud_Media'] - tendencia_anual['Magnitud_Std'],
                         tendencia_anual['Magnitud_Media'] + tendencia_anual['Magnitud_Std'],
                         alpha=0.2, color='mediumseagreen', label='±1 Desv. Est.')
axes[1, 0].set_title('Magnitud Media Anual (±1σ)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Año')
axes[1, 0].set_ylabel('Magnitud MW')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Gráfico 4: Profundidad media anual
axes[1, 1].plot(tendencia_anual['Año'], tendencia_anual['Profundidad_Media'], marker='^', linewidth=2, markersize=7, color='mediumpurple')
axes[1, 1].set_title('Profundidad Media Anual', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Año')
axes[1, 1].set_ylabel('Profundidad (km)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('series_temporales_anuales.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Series temporales generadas')

# **PARTE 6: ANÁLISIS GEOESPACIAL DE EPICENTROS**

## **6.1 Mapa Interactivo de Epicentros**

In [ ]:
# ============================================================================
# MAPAS INTERACTIVOS: EPICENTROS
# ============================================================================

# Centro del mapa (Chile)
centro_lat = (df['LATITUD'].max() + df['LATITUD'].min()) / 2
centro_lon = (df['LONGITUD'].max() + df['LONGITUD'].min()) / 2

print(f'Creando mapa interactivo centrado en: ({centro_lat:.2f}, {centro_lon:.2f})')

# Crear mapa
mapa = folium.Map(location=[centro_lat, centro_lon], zoom_start=5, tiles='OpenStreetMap')

# Agregar heatmap de epicentros
heat_data = [[row['LATITUD'], row['LONGITUD'], row['MAGNITUD_MW']/10] 
             for idx, row in df.sample(n=min(5000, len(df))).iterrows()]
HeatMap(heat_data, radius=15, blur=25, max_zoom=1).add_to(mapa)

# Agregar cluster de marcadores para eventos de alta magnitud
marker_cluster = MarkerCluster().add_to(mapa)
for idx, row in df[df['MAGNITUD_MW'] >= 6.0].iterrows():
    folium.CircleMarker(
        location=[row['LATITUD'], row['LONGITUD']],
        radius=5 * (row['MAGNITUD_MW'] - 5),
        popup=f"Mw {row['MAGNITUD_MW']:.1f}, {row['PROFUNDIDAD_KM']:.0f} km, {row['ANIO']:.0f}",
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.7,
        weight=2
    ).add_to(mapa)

# Guardar mapa
mapa_path = 'mapa_epicentros.html'
mapa.save(mapa_path)
print(f'✓ Mapa interactivo guardado: {mapa_path}')
print('\nGráficos geoespaciales:')
print(f'  - Total de epicentros: {len(df):,}')
print(f'  - Eventos con Mw ≥ 6.0: {len(df[df["MAGNITUD_MW"] >= 6.0]):,}')

## **6.2 Scatter Plot Geográfico**

In [ ]:
# ============================================================================
# SCATTER PLOT: DISTRIBUCIÓN GEOGRÁFICA
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Gráfico 1: Epicentros coloreados por magnitud
scatter1 = axes[0].scatter(df['LONGITUD'], df['LATITUD'], 
                           c=df['MAGNITUD_MW'], cmap='Reds', s=50, alpha=0.6,
                           edgecolors='black', linewidth=0.5, vmin=2, vmax=8)
axes[0].set_title('Distribución Geográfica de Epicentros (coloreado por magnitud)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Longitud (°W)')
axes[0].set_ylabel('Latitud (°S)')
axes[0].grid(True, alpha=0.3)
cbar1 = plt.colorbar(scatter1, ax=axes[0])
cbar1.set_label('Magnitud MW')

# Gráfico 2: Epicentros coloreados por profundidad
scatter2 = axes[1].scatter(df['LONGITUD'], df['LATITUD'],
                           c=df['PROFUNDIDAD_KM'], cmap='viridis', s=50, alpha=0.6,
                           edgecolors='black', linewidth=0.5)
axes[1].set_title('Distribución Geográfica de Epicentros (coloreado por profundidad)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Longitud (°W)')
axes[1].set_ylabel('Latitud (°S)')
axes[1].grid(True, alpha=0.3)
cbar2 = plt.colorbar(scatter2, ax=axes[1])
cbar2.set_label('Profundidad (km)')

plt.tight_layout()
plt.savefig('distribucion_geografica.png', dpi=150, bbox_inches='tight')
plt.show()

print('✓ Gráficos geográficos generados')

# **PARTE 7: PERFILES DE RIESGO SÍSMICO POR REGIÓN**

## **7.1 Segmentación Regional**

In [ ]:
# ============================================================================
# PERFILES DE RIESGO POR REGIÓN
# ============================================================================

# Definir regiones según latitud (división administrativa simplificada)
def asignar_region(latitud):
    if latitud > -27:
        return 'Norte'
    elif latitud > -37:
        return 'Centro'
    elif latitud > -45:
        return 'Sur'
    else:
        return 'Zona Austral'

df['REGION'] = df['LATITUD'].apply(asignar_region)

# Calcular indicadores por región
perfiles_region = df.groupby('REGION').agg({
    'MAGNITUD_MW': ['count', 'mean', 'max', 'std'],
    'PROFUNDIDAD_KM': 'mean',
    'ANIO': ['min', 'max'],
    'LATITUD': 'mean',
    'LONGITUD': 'mean'
}).reset_index()

perfiles_region.columns = ['Región', 'Total_Eventos', 'Mag_Media', 'Mag_Max', 'Mag_Std',
                           'Prof_Media', 'Año_Inicio', 'Año_Fin', 'Lat_Centro', 'Lon_Centro']

print('PERFILES DE RIESGO SÍSMICO POR REGIÓN')
print('=' * 100)
print(perfiles_region.to_string(index=False))

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Gráfico 1: Eventos por región
colores_region = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
axes[0, 0].bar(perfiles_region['Región'], perfiles_region['Total_Eventos'], 
               color=colores_region, edgecolor='black', alpha=0.8)
axes[0, 0].set_title('Número Total de Eventos por Región', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Cantidad de Eventos')
axes[0, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(perfiles_region['Total_Eventos']):
    axes[0, 0].text(i, v + 500, f'{int(v):,}', ha='center', va='bottom', fontweight='bold')

# Gráfico 2: Magnitud máxima por región
axes[0, 1].bar(perfiles_region['Región'], perfiles_region['Mag_Max'],
               color=colores_region, edgecolor='black', alpha=0.8)
axes[0, 1].set_title('Magnitud Máxima por Región', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Magnitud MW')
axes[0, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(perfiles_region['Mag_Max']):
    axes[0, 1].text(i, v + 0.1, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

# Gráfico 3: Magnitud media por región
axes[1, 0].bar(perfiles_region['Región'], perfiles_region['Mag_Media'],
               color=colores_region, edgecolor='black', alpha=0.8)
axes[1, 0].set_title('Magnitud Media por Región', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Magnitud MW')
axes[1, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(perfiles_region['Mag_Media']):
    axes[1, 0].text(i, v + 0.02, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

# Gráfico 4: Profundidad media por región
axes[1, 1].bar(perfiles_region['Región'], perfiles_region['Prof_Media'],
               color=colores_region, edgecolor='black', alpha=0.8)
axes[1, 1].set_title('Profundidad Media por Región', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Profundidad (km)')
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(perfiles_region['Prof_Media']):
    axes[1, 1].text(i, v + 1, f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('perfiles_region.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Perfiles de riesgo regional generados')

# **PARTE 8: MODELOS PREDICTIVOS DE MAGNITUD**

## **8.1 Preparación de Datos para ML**

In [ ]:
# ============================================================================
# PREPARACIÓN DE DATOS PARA MACHINE LEARNING
# ============================================================================

# Seleccionar variables para modelado
X_features = ['PROFUNDIDAD_KM', 'LATITUD', 'LONGITUD', 'ANIO', 'MES', 'DIA']
y_target = 'MAGNITUD_MW'

# Crear dataset limpio
df_ml = df[X_features + [y_target]].dropna()

print(f'Dataset para ML:')
print(f'  - Registros: {len(df_ml):,}')
print(f'  - Features: {len(X_features)}')
print(f'  - Target (Magnitud MW): rango [{df_ml[y_target].min():.2f}, {df_ml[y_target].max():.2f}]')

# Separar features y target
X = df_ml[X_features]
y = df_ml[y_target]

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\n  - Train: {len(X_train):,} registros')
print(f'  - Test: {len(X_test):,} registros')
print(f'\n✓ Datos preparados para modelado')

## **8.2 Entrenamiento de Modelos**

In [ ]:
# ============================================================================
# ENTRENAMIENTO Y EVALUACIÓN DE MODELOS
# ============================================================================

resultados_modelos = []

print('ENTRENAMIENTO DE MODELOS DE REGRESIÓN')
print('=' * 100)

# 1. Regresión Lineal
print('\n1. REGRESIÓN LINEAL')
model_lr = LinearRegression()
model_lr.fit(X_train_scaled, y_train)
y_pred_lr = model_lr.predict(X_test_scaled)

r2_lr = r2_score(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print(f'  - R²: {r2_lr:.4f}')
print(f'  - MAE: {mae_lr:.4f}')
print(f'  - RMSE: {rmse_lr:.4f}')

resultados_modelos.append({
    'Modelo': 'Regresión Lineal',
    'R²': r2_lr,
    'MAE': mae_lr,
    'RMSE': rmse_lr
})

# 2. Random Forest
print('\n2. RANDOM FOREST')
model_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, max_depth=15)
model_rf.fit(X_train_scaled, y_train)
y_pred_rf = model_rf.predict(X_test_scaled)

r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f'  - R²: {r2_rf:.4f}')
print(f'  - MAE: {mae_rf:.4f}')
print(f'  - RMSE: {rmse_rf:.4f}')

resultados_modelos.append({
    'Modelo': 'Random Forest',
    'R²': r2_rf,
    'MAE': mae_rf,
    'RMSE': rmse_rf
})

# 3. Gradient Boosting
print('\n3. GRADIENT BOOSTING')
model_gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
model_gb.fit(X_train_scaled, y_train)
y_pred_gb = model_gb.predict(X_test_scaled)

r2_gb = r2_score(y_test, y_pred_gb)
mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))

print(f'  - R²: {r2_gb:.4f}')
print(f'  - MAE: {mae_gb:.4f}')
print(f'  - RMSE: {rmse_gb:.4f}')

resultados_modelos.append({
    'Modelo': 'Gradient Boosting',
    'R²': r2_gb,
    'MAE': mae_gb,
    'RMSE': rmse_gb
})

# Tabla comparativa
df_resultados = pd.DataFrame(resultados_modelos)
print('\n\nRESUMEN COMPARATIVO DE MODELOS')
print('=' * 100)
print(df_resultados.to_string(index=False))

print(f'\n✓ Modelos entrenados exitosamente')

## **8.3 Feature Importance y Análisis de Modelos**

In [ ]:
# ============================================================================
# ANÁLISIS DE FEATURE IMPORTANCE
# ============================================================================

# Feature importance de Random Forest
feature_importance_rf = pd.DataFrame({
    'Feature': X_features,
    'Importance': model_rf.feature_importances_
}).sort_values('Importance', ascending=False)

print('IMPORTANCIA DE FEATURES - RANDOM FOREST')
print('=' * 80)
print(feature_importance_rf.to_string(index=False))

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Gráfico 1: Feature Importance
axes[0, 0].barh(feature_importance_rf['Feature'], feature_importance_rf['Importance'], color='steelblue', edgecolor='black')
axes[0, 0].set_title('Feature Importance - Random Forest', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Importancia')
axes[0, 0].invert_yaxis()
axes[0, 0].grid(True, alpha=0.3, axis='x')

# Gráfico 2: Predicciones vs Reales (Random Forest)
axes[0, 1].scatter(y_test, y_pred_rf, alpha=0.5, s=20, edgecolors='black', linewidth=0.5)
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Predicción Perfecta')
axes[0, 1].set_title(f'Random Forest: Predicción vs Real (R²={r2_rf:.4f})', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Magnitud Real')
axes[0, 1].set_ylabel('Magnitud Predicha')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Gráfico 3: Residuos (Random Forest)
residuos_rf = y_test - y_pred_rf
axes[1, 0].scatter(y_pred_rf, residuos_rf, alpha=0.5, s=20, edgecolors='black', linewidth=0.5)
axes[1, 0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 0].set_title('Residuos - Random Forest', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Magnitud Predicha')
axes[1, 0].set_ylabel('Residuos')
axes[1, 0].grid(True, alpha=0.3)

# Gráfico 4: Comparación de métricas
modelos_nombres = df_resultados['Modelo']
r2_scores = df_resultados['R²']
colores_modelo = ['#1f77b4', '#ff7f0e', '#2ca02c']
axes[1, 1].bar(modelos_nombres, r2_scores, color=colores_modelo, edgecolor='black', alpha=0.8)
axes[1, 1].set_title('Comparación de R² entre Modelos', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('R² Score')
axes[1, 1].set_ylim([0, 1])
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(r2_scores):
    axes[1, 1].text(i, v + 0.02, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('analisis_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Análisis de modelos generado')

# **PARTE 9: DASHBOARD DE MÉTRICAS CLAVE**

## **9.1 Indicadores Principales (KPIs)**

In [ ]:
# ============================================================================
# DASHBOARD DE KPIS
# ============================================================================

print('\n' + '=' * 100)
print(' ' * 35 + 'DASHBOARD DE MÉTRICAS CLAVE - FASE 4')
print('=' * 100)

# Calcular KPIs
total_eventos = len(df)
mag_promedio = df['MAGNITUD_MW'].mean()
mag_maxima = df['MAGNITUD_MW'].max()
prof_promedio = df['PROFUNDIDAD_KM'].mean()
evento_mas_fuerte = df.loc[df['MAGNITUD_MW'].idxmax()]
region_mas_activa = df['REGION'].value_counts().idxmax()
evento_mas_fuerte_anio = df.loc[df['ANIO'].idxmax()]

print(f'\n📊 ESTADÍSTICAS GENERALES')
print(f'{'─' * 100}')
print(f'  Total de eventos registrados: {total_eventos:,}')
print(f'  Período analizado: {int(df["ANIO"].min())} - {int(df["ANIO"].max())}')
print(f'  Promedio de eventos/año: {total_eventos / (df["ANIO"].max() - df["ANIO"].min() + 1):.0f}')

print(f'\n📈 MAGNITUD')
print(f'{'─' * 100}')
print(f'  Magnitud promedio: {mag_promedio:.3f} Mw')
print(f'  Magnitud máxima: {mag_maxima:.3f} Mw')
print(f'  Magnitud mínima: {df["MAGNITUD_MW"].min():.3f} Mw')
print(f'  Desviación estándar: {df["MAGNITUD_MW"].std():.3f} Mw')

print(f'\n🌍 PROFUNDIDAD')
print(f'{'─' * 100}')
print(f'  Profundidad promedio: {prof_promedio:.2f} km')
print(f'  Profundidad máxima: {df["PROFUNDIDAD_KM"].max():.2f} km')
print(f'  Profundidad mínima: {df["PROFUNDIDAD_KM"].min():.2f} km')
print(f'  Eventos superficiales (<70 km): {len(df[df["PROFUNDIDAD_KM"] < 70]):,} ({100*len(df[df["PROFUNDIDAD_KM"] < 70])/len(df):.1f}%)')

print(f'\n🎯 EVENTO MÁS FUERTE')
print(f'{'─' * 100}')
print(f'  Magnitud: {evento_mas_fuerte["MAGNITUD_MW"]:.3f} Mw')
print(f'  Profundidad: {evento_mas_fuerte["PROFUNDIDAD_KM"]:.2f} km')
print(f'  Fecha: {evento_mas_fuerte["FECHA_UTC"]}')
print(f'  Ubicación: ({evento_mas_fuerte["LATITUD"]:.2f}°S, {evento_mas_fuerte["LONGITUD"]:.2f}°W)')

print(f'\n🗺️ REGIÓN MÁS ACTIVA')
print(f'{'─' * 100}')
region_data = df[df['REGION'] == region_mas_activa]
print(f'  Región: {region_mas_activa}')
print(f'  Eventos: {len(region_data):,}')
print(f'  Magnitud media: {region_data["MAGNITUD_MW"].mean():.3f} Mw')
print(f'  Magnitud máxima: {region_data["MAGNITUD_MW"].max():.3f} Mw')

print(f'\n⚠️ INDICADORES DE ALERTA')
print(f'{'─' * 100}')
evt_mw7_plus = len(df[df['MAGNITUD_MW'] >= 7.0])
evt_mw6_plus = len(df[df['MAGNITUD_MW'] >= 6.0])
evt_mw5_plus = len(df[df['MAGNITUD_MW'] >= 5.0])
print(f'  Eventos con Mw ≥ 7.0: {evt_mw7_plus} (CRÍTICO)')
print(f'  Eventos con Mw ≥ 6.0: {evt_mw6_plus} (ALTO RIESGO)')
print(f'  Eventos con Mw ≥ 5.0: {evt_mw5_plus} (MODERADO RIESGO)')

print(f'\n🔮 RENDIMIENTO DEL MEJOR MODELO')
print(f'{'─' * 100}')
best_model_idx = df_resultados['R²'].idxmax()
best_model = df_resultados.iloc[best_model_idx]
print(f'  Modelo: {best_model["Modelo"]}')
print(f'  R²: {best_model["R²"]:.4f}')
print(f'  MAE: {best_model["MAE"]:.4f} Mw')
print(f'  RMSE: {best_model["RMSE"]:.4f} Mw')

print(f'\n' + '=' * 100)
print('\n✓ Dashboard completado')

## **9.2 Visualización del Dashboard**

In [ ]:
# ============================================================================
# VISUALIZACIÓN DEL DASHBOARD
# ============================================================================

fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# KPI 1: Total de eventos
ax1 = fig.add_subplot(gs[0, 0])
ax1.text(0.5, 0.7, f'{len(df):,}', ha='center', va='center', fontsize=40, fontweight='bold', color='steelblue')
ax1.text(0.5, 0.3, 'Total de Eventos', ha='center', va='center', fontsize=12, fontweight='bold')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')
ax1.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=False, edgecolor='steelblue', linewidth=3))

# KPI 2: Magnitud promedio
ax2 = fig.add_subplot(gs[0, 1])
ax2.text(0.5, 0.7, f'{mag_promedio:.2f}', ha='center', va='center', fontsize=40, fontweight='bold', color='coral')
ax2.text(0.5, 0.3, 'Magnitud Promedio (Mw)', ha='center', va='center', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')
ax2.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=False, edgecolor='coral', linewidth=3))

# KPI 3: Magnitud máxima
ax3 = fig.add_subplot(gs[0, 2])
ax3.text(0.5, 0.7, f'{mag_maxima:.2f}', ha='center', va='center', fontsize=40, fontweight='bold', color='red')
ax3.text(0.5, 0.3, 'Magnitud Máxima (Mw)', ha='center', va='center', fontsize=12, fontweight='bold')
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.axis('off')
ax3.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=False, edgecolor='red', linewidth=3))

# Gráfico 4: Distribución por magnitud
ax4 = fig.add_subplot(gs[1, 0])
df['CATEGORIA_MAGNITUD'].value_counts().plot(kind='barh', ax=ax4, color='viridis')
ax4.set_title('Distribución por Categoría de Magnitud', fontsize=11, fontweight='bold')
ax4.set_xlabel('Cantidad')
ax4.grid(True, alpha=0.3, axis='x')

# Gráfico 5: Eventos por región
ax5 = fig.add_subplot(gs[1, 1])
df['REGION'].value_counts().plot(kind='pie', ax=ax5, autopct='%1.1f%%', colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
ax5.set_title('Distribución de Eventos por Región', fontsize=11, fontweight='bold')
ax5.set_ylabel('')

# Gráfico 6: Tendencia anual
ax6 = fig.add_subplot(gs[1, 2])
tendencia_anual.plot(x='Año', y='Eventos', ax=ax6, marker='o', linewidth=2, markersize=5, color='mediumseagreen')
ax6.set_title('Tendencia Anual de Eventos', fontsize=11, fontweight='bold')
ax6.set_xlabel('Año')
ax6.set_ylabel('Eventos')
ax6.grid(True, alpha=0.3)
ax6.legend(['Eventos anuales'])

# Gráfico 7: Top eventos por magnitud
ax7 = fig.add_subplot(gs[2, :])
top_eventos = df.nlargest(10, 'MAGNITUD_MW')[['MAGNITUD_MW', 'PROFUNDIDAD_KM', 'ANIO', 'REGION']].reset_index(drop=True)
ax7.axis('off')
table_data = []
for idx, row in top_eventos.iterrows():
    table_data.append([f"{idx+1}", f"{row['MAGNITUD_MW']:.2f}", f"{row['PROFUNDIDAD_KM']:.0f} km", f"{int(row['ANIO'])}", row['REGION']])

table = ax7.table(cellText=table_data,
                  colLabels=['#', 'Magnitud (Mw)', 'Profundidad', 'Año', 'Región'],
                  cellLoc='center',
                  loc='center',
                  bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)
# Colorear encabezado
for i in range(5):
    table[(0, i)].set_facecolor('#4ECDC4')
    table[(0, i)].set_text_props(weight='bold', color='white')
# Colorear filas alternadas
for i in range(1, len(table_data) + 1):
    color = '#f0f0f0' if i % 2 == 0 else 'white'
    for j in range(5):
        table[(i, j)].set_facecolor(color)

ax7.text(0.5, 1.15, 'TOP 10 EVENTOS MÁS FUERTES', ha='center', fontsize=12, fontweight='bold', transform=ax7.transAxes)

plt.suptitle('DASHBOARD DE ANÁLISIS SÍSMICO - CHILE 2000-2024', fontsize=16, fontweight='bold', y=0.995)
plt.savefig('dashboard_kpis.png', dpi=150, bbox_inches='tight')
plt.show()

print('✓ Dashboard de KPIs generado')

# **CONCLUSIONES Y SÍNTESIS**

## **Hallazgos Principales**

In [ ]:
print('\n' + '=' * 100)
print(' ' * 30 + 'CONCLUSIONES DEL ANÁLISIS AVANZADO - FASE 4')
print('=' * 100)

print(f'\n📊 ANÁLISIS EXPLORATORIO')
print(f'{'─' * 100}')
print(f'  ✓ Distribución de magnitudes: Sesgada a la derecha (mayor cantidad de sismos pequeños)')
print(f'  ✓ Profundidad promedio: {prof_promedio:.1f} km (mayoría superficiales < 70 km)')
print(f'  ✓ Correlación magnitud-profundidad: Débil ({df[["MAGNITUD_MW", "PROFUNDIDAD_KM"]].corr().iloc[0,1]:.3f})')

print(f'\n📈 SERIES TEMPORALES')
print(f'{'─' * 100}')
trend_slope = np.polyfit(tendencia_anual['Año'], tendencia_anual['Eventos'], 1)[0]
print(f'  ✓ Tendencia de eventos: {"↑ Creciente" if trend_slope > 0 else "↓ Decreciente"} ({trend_slope:+.1f} eventos/año)')
print(f'  ✓ Atribuible a: Mejora tecnológica en redes sísmológicas')
print(f'  ✓ Años con mayor actividad: {tendencia_anual.nlargest(3, "Eventos")["Año"].values}')

print(f'\n🗺️ ANÁLISIS GEOESPACIAL')
print(f'{'─' * 100}')
print(f'  ✓ Zona de mayor concentración: Región Centro y Zona de Subducción')
print(f'  ✓ Eventos Mw ≥ 6.0: {len(df[df["MAGNITUD_MW"] >= 6.0])} registrados')
print(f'  ✓ Relación con tectónica: Clara correlación con límite de placas Nazca-Sudamericana')

print(f'\n⚠️ PERFILES DE RIESGO REGIONAL')
print(f'{'─' * 100}')
for _, row in perfiles_region.iterrows():
    riesgo = '🔴 ALTO' if row['Mag_Max'] >= 7.0 else '🟠 MODERADO' if row['Mag_Max'] >= 6.0 else '🟡 BAJO'
    print(f'  {riesgo} {row["Región"]}: {int(row["Total_Eventos"]):,} eventos, Mag_max={row["Mag_Max"]:.2f}')

print(f'\n🔮 MODELOS PREDICTIVOS')
print(f'{'─' * 100}')
print(f'  ✓ Mejor modelo: {best_model["Modelo"]} (R²={best_model["R²"]:.4f})')
print(f'  ✓ Error promedio: {best_model["MAE"]:.3f} Mw')
print(f'  ✓ Feature más importante: Profundidad')
print(f'  ✓ Aplicación: Predicción de magnitud a partir de profundidad y ubicación')

print(f'\n✅ RECOMENDACIONES')
print(f'{'─' * 100}')
print(f'  1. Implementar sistema de alerta temprana para eventos Mw ≥ 6.0 en zona central')
print(f'  2. Enfocar planes de emergencia en regiones Centro y Sur (mayor densidad de epicentros)')
print(f'  3. Utilizar modelos ML para predicción de magnitud en tiempo real')
print(f'  4. Actualizar códigos de construcción sismo-resistente según riesgo regional')
print(f'  5. Continuar mejora de redes sísmológicas para captura de eventos pequeños')

print(f'\n' + '=' * 100)
print('\n✓ ANÁLISIS FASE 4 COMPLETADO EXITOSAMENTE')
print('\nArchivos generados:')
print('  - analisis_magnitud.png')
print('  - analisis_profundidad.png')
print('  - matriz_correlaciones.png')
print('  - scatter_magnitud_profundidad.png')
print('  - series_temporales_anuales.png')
print('  - mapa_epicentros.html')
print('  - distribucion_geografica.png')
print('  - perfiles_region.png')
print('  - analisis_modelos.png')
print('  - dashboard_kpis.png')